In [ ]:
# Цель:
# 1) показать dense retrieval на маленьком примере
# 2) применить multilingual-e5-large к chunks_final.csv и gold_final.csv
# 3) сохранить chunks_with_e5_embeddings.csv
# 4) сохранить dense_rankings.csv

# Dense retrieval

В dense retrieval тексты и вопросы переводятся в числовые векторы (embeddings).

Идея очень простая:

- похожие по смыслу тексты должны иметь похожие векторы;
- вопрос тоже переводится в вектор;
- дальше мы ищем чанки, чьи векторы ближе всего к вектору вопроса.

В этом ноутбуке мы используем модель `intfloat/multilingual-e5-large`.

Важно:
- для запроса нужно добавлять префикс `query: `
- для чанка нужно добавлять префикс `passage: `
- это чисто нюанс работы с конкретной моделью


# Почему выбрана multilingual-e5-large

Мы используем `intfloat/multilingual-e5-large`, потому что:

- это retrieval-oriented embedding model;
- она поддерживает мультиязычные тексты;
- для неё есть простой и понятный рецепт применения к retrieval-задачам;
- для нашего проекта важнее воспроизводимый baseline, чем подбор большого числа моделей.

Ограничение:
- длинные тексты обрезаются до 512 токенов.
- для наших коротких чанков это допустимо.

In [1]:
!pip install -U sentence-transformers transformers pandas numpy scikit-learn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 96.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 105.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 73.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 102.9 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.2
    Uninstalling pandas-2.2.2:
      Successfully uninstalled pandas-2.2.2
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
  

In [3]:
# После установки пакетов перезапустите среду. Не забудьте выбрать GPU
import json
import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [4]:
MODEL_NAME = "intfloat/multilingual-e5-large"

CHUNKS_PATH = "chunks_final.csv"
GOLD_PATH = "gold_final.csv"

CHUNKS_WITH_EMB_PATH = "chunks_with_e5_embeddings.csv"
DENSE_RANKINGS_PATH = "dense_rankings.csv"

TOP_K = 10
BATCH_SIZE = 32

Демо:

In [5]:
toy_chunks = [
    "Выручка компании за 2024 год составила 703 741 млн руб.",
    "Чистая прибыль выросла на 15 процентов.",
    "Капитальные затраты снизились по сравнению с прошлым годом.",
    "Количество клиентов мобильной связи увеличилось."
]

toy_chunk_ids = ["d1", "d2", "d3", "d4"]

toy_question = "Какая выручка компании за 2024 год?"

In [6]:
toy_model = SentenceTransformer(MODEL_NAME)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/690 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/418 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/201 [00:00<?, ?B/s]

In [7]:
toy_passages = [f"passage: {text}" for text in toy_chunks]
toy_query = f"query: {toy_question}"

print(toy_query)
print(toy_passages[0])

query: Какая выручка компании за 2024 год?
passage: Выручка компании за 2024 год составила 703 741 млн руб.


In [8]:
toy_passage_embeddings = toy_model.encode(
    toy_passages,
    batch_size=BATCH_SIZE,
    normalize_embeddings=True,
    show_progress_bar=False,
)

toy_query_embedding = toy_model.encode(
    [toy_query],
    batch_size=1,
    normalize_embeddings=True,
    show_progress_bar=False,
)

In [9]:
# 1 вектор на 1024 элемента
toy_query_embedding

array([[-0.00502959,  0.001259  , -0.01846605, ...,  0.00483143,
        -0.04279127,  0.00040075]], shape=(1, 1024), dtype=float32)

In [10]:
# 4 вектора по 1024 элемена для 4 вопросов
toy_passage_embeddings

array([[ 0.01627781, -0.00793281, -0.01652344, ...,  0.01013484,
        -0.04524498,  0.00318337],
       [ 0.0291121 , -0.01273165, -0.0110264 , ..., -0.00083146,
        -0.02153984, -0.00045761],
       [ 0.02496765, -0.00453519, -0.02472969, ...,  0.00524633,
        -0.04414998, -0.01316174],
       [ 0.01591744, -0.02578917, -0.02609371, ..., -0.00090834,
        -0.01726528,  0.00681287]], shape=(4, 1024), dtype=float32)

In [11]:
# Формулу расчета найдите сами, она простая. Вставьте в отчет
toy_scores = cosine_similarity(toy_query_embedding, toy_passage_embeddings)[0]

toy_results = pd.DataFrame({
    "chunk_id": toy_chunk_ids,
    "text": toy_chunks,
    "score": toy_scores,
}).sort_values("score", ascending=False).reset_index(drop=True)

toy_results

,chunk_id,text,score
0,d1,Выручка компании за 2024 год составила 703 741...,0.908400
1,d3,Капитальные затраты снизились по сравнению с п...,0.736221
2,d2,Чистая прибыль выросла на 15 процентов.,0.732288
3,d4,Количество клиентов мобильной связи увеличилось.,0.709866


# Что произошло

Мы:

1. перевели каждый чанк в embedding;
2. перевели вопрос в embedding;
3. посчитали cosine similarity;
4. отсортировали чанки по близости к вопросу.

Чанк с максимальным score должен быть наиболее релевантным по смыслу.

В принципе это всё что нам было нужно. Далее надо подгрузить реальные данные

In [ ]:
chunks_df = pd.read_csv(CHUNKS_PATH)
gold_df = pd.read_csv(GOLD_PATH)

print("chunks_df:", chunks_df.shape)
print("gold_df:", gold_df.shape)

display(chunks_df.head())
display(gold_df.head())

In [ ]:
required_chunk_cols = ["chunk_id", "text"]
required_gold_cols = ["query_id", "question", "relevant_chunk_id"]

missing_chunk_cols = [c for c in required_chunk_cols if c not in chunks_df.columns]
missing_gold_cols = [c for c in required_gold_cols if c not in gold_df.columns]

assert not missing_chunk_cols, f"В chunks_final.csv не хватает колонок: {missing_chunk_cols}"
assert not missing_gold_cols, f"В gold_final.csv не хватает колонок: {missing_gold_cols}"

chunks_df["chunk_id"] = chunks_df["chunk_id"].astype(str).str.strip()
chunks_df["text"] = chunks_df["text"].fillna("").astype(str).str.strip()

gold_df["query_id"] = gold_df["query_id"].astype(str).str.strip()
gold_df["question"] = gold_df["question"].fillna("").astype(str).str.strip()
gold_df["relevant_chunk_id"] = gold_df["relevant_chunk_id"].astype(str).str.strip()

chunks_df = chunks_df[chunks_df["text"] != ""].reset_index(drop=True)

print("Проверка входных данных пройдена")

# Задание

Нужно реализовать dense retrieval на наших данных.

Что нужно сделать:
1. получить embeddings для всех чанков;
2. сохранить chunks_with_e5_embeddings.csv;
3. получить embeddings для всех вопросов;
4. для каждого вопроса найти top-k чанков по cosine similarity;
5. сохранить dense_rankings.csv.

Формат dense_rankings.csv:
- query_id
- method
- rank
- chunk_id
- score

Между 2 и 3 пунктом вполне может быть перерыв, тогда просто подгрузите датасет с эмбеддингами

In [ ]:
model = SentenceTransformer(MODEL_NAME) # Та же модель. В целом можно сразу назвать model и не грузить два раза

In [ ]:
passage_texts = [f"passage: {text}" for text in chunks_df["text"].tolist()]
query_texts = [f"query: {text}" for text in gold_df["question"].tolist()]

print(passage_texts[0][:200])
print(query_texts[0])

In [ ]:
def encode_texts(model, texts, batch_size=32):
    """
    Кодирует список строк в embeddings.

    Аргументы:
    - model: загруженная SentenceTransformer-модель
    - texts: список строк
    - batch_size: размер батча при кодировании

    Возвращает:
    - numpy array формы (n_texts, embedding_dim)

    Что нужно сделать:
    - вызвать model.encode(...)
    - включить normalize_embeddings=True
    - вернуть результат
    """
    # TODO: реализовать
    raise NotImplementedError

# Может работать какое-то время, не моментально. Можете добавить базовую observability с tqdm чтобы видеть прогресс. Вообще хорошая практика для каких-то долгих задач.
chunk_embeddings = encode_texts(model, passage_texts, batch_size=BATCH_SIZE)
print(chunk_embeddings.shape)

In [ ]:
# Добавляем эмбеддинги для чанков в датафрейм
chunks_with_emb_df = chunks_df.copy()
chunks_with_emb_df["embedding"] = [
    json.dumps(vec.tolist(), ensure_ascii=False) for vec in chunk_embeddings
]

chunks_with_emb_df.head()

In [ ]:
# И сохраняем
chunks_with_emb_df.to_csv(CHUNKS_WITH_EMB_PATH, index=False, encoding="utf-8-sig")
print(f"Сохранён файл: {CHUNKS_WITH_EMB_PATH}")

In [ ]:
# Теперь вопросы
# В реальной эксплуатации запрос юзера векторизуется в реальном времени. У нас на golden set векторизуем сразу все
query_embeddings = encode_texts(model, query_texts, batch_size=BATCH_SIZE)
print(query_embeddings.shape)

In [ ]:
def retrieve_top_k_dense(query_id, query_embedding, chunk_embeddings, chunks_df, top_k=10):
    """
    Строит top-k выдачу для одного вопроса.

    Аргументы:
    - query_id: id вопроса
    - query_embedding: embedding одного вопроса
    - chunk_embeddings: embeddings всех чанков
    - chunks_df: DataFrame с чанками, где есть колонка chunk_id
    - top_k: сколько лучших чанков вернуть

    Возвращает:
    DataFrame с колонками:
    - query_id
    - method
    - rank
    - chunk_id
    - score

    Что нужно сделать:
    1. посчитать cosine similarity между query_embedding и всеми chunk_embeddings
    2. отсортировать чанки по убыванию score
    3. взять top_k лучших
    4. собрать результат в DataFrame
    """
    # TODO: реализовать
    raise NotImplementedError

In [ ]:
test_query_id = gold_df.iloc[0]["query_id"]
test_query_embedding = query_embeddings[0]

test_results = retrieve_top_k_dense(
    query_id=test_query_id,
    query_embedding=test_query_embedding,
    chunk_embeddings=chunk_embeddings,
    chunks_df=chunks_df,
    top_k=TOP_K,
)

display(test_results)

In [ ]:
def build_dense_rankings(gold_df, query_embeddings, chunk_embeddings, chunks_df, top_k=10):
    """
    Строит итоговый ranking DataFrame для всех вопросов.

    Аргументы:
    - gold_df: таблица вопросов, где есть query_id
    - query_embeddings: embeddings всех вопросов в том же порядке, что и строки gold_df
    - chunk_embeddings: embeddings всех чанков
    - chunks_df: таблица чанков
    - top_k: сколько чанков возвращать на каждый вопрос

    Возвращает:
    единый DataFrame со всеми результатами retrieval

    Что нужно сделать:
    1. пройти по всем вопросам из gold_df
    2. для каждого вопроса вызвать retrieve_top_k_dense(...)
    3. склеить результаты в один DataFrame
    """
    # TODO: реализовать. Тоже советую юзать tqdm
    raise NotImplementedError

In [ ]:
dense_rankings_df = build_dense_rankings(
    gold_df=gold_df,
    query_embeddings=query_embeddings,
    chunk_embeddings=chunk_embeddings,
    chunks_df=chunks_df,
    top_k=TOP_K,
)

print("dense_rankings_df:", dense_rankings_df.shape)
dense_rankings_df.head(20)

In [ ]:
# проверим
required_output_cols = ["query_id", "method", "rank", "chunk_id", "score"]
missing_output_cols = [c for c in required_output_cols if c not in dense_rankings_df.columns]
assert not missing_output_cols, f"В dense_rankings_df не хватает колонок: {missing_output_cols}"

assert (dense_rankings_df["method"] == "dense").all(), "Колонка method должна быть равна 'dense'"
assert (dense_rankings_df["rank"] >= 1).all(), "rank должен начинаться с 1"
assert dense_rankings_df["chunk_id"].notna().all(), "Есть пустые chunk_id"
assert dense_rankings_df["query_id"].notna().all(), "Есть пустые query_id"

print("Проверка dense_rankings_df пройдена")

In [ ]:
# Посмотрим
sample_query_ids = gold_df["query_id"].head(3).tolist()

for qid in sample_query_ids:
    print("=" * 80)
    print("QUERY_ID:", qid)
    print("QUESTION:", gold_df.loc[gold_df["query_id"] == qid, "question"].iloc[0])
    display(dense_rankings_df[dense_rankings_df["query_id"] == qid].head(10))

In [ ]:
# Сохраним
dense_rankings_df.to_csv(DENSE_RANKINGS_PATH, index=False, encoding="utf-8-sig")
print(f"Сохранён файл: {DENSE_RANKINGS_PATH}")